# 🤸 เทรน G1 ทรงตัว (Balance) บน Colab — ได้โมเดล .pt

notebook นี้เทรน G1 ให้ **ยืนทรงตัวไม่ล้ม** (ต่างจาก train_g1_colab ที่เทรนให้ **เดิน**)

**Balance task ต่างจาก velocity ยังไง:**
- ทุก environment ถูกสั่งให้ **ยืนนิ่ง** (ความเร็วเป้าหมาย = 0)
- reward เน้น **ตั้งตัวตรง (upright) + รักษาท่ายืน (pose)** แทนการเดินตามคำสั่ง
- ตัด reward การเดินออก (track velocity = 0)

**⚠️ ก่อนเริ่ม:** `Runtime → Change runtime type → T4 GPU`

งานทรงตัวง่ายกว่าการเดิน — เห็นผลไวกว่า (หุ่นเริ่มยืนนิ่งได้ในไม่กี่ร้อยรอบ)

## 1) ตรวจว่ามี GPU

ถ้าบรรทัดล่างไม่ขึ้นชื่อ GPU (เช่น Tesla T4) ให้กลับไปตั้ง Runtime ก่อน

In [6]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


## 2) ติดตั้ง mjlab

clone repo แล้วติดตั้งแบบ editable (`-e`) จะได้แก้โค้ด/เพิ่ม task ได้
(ใช้เวลาสักครู่)

In [7]:
# clone repo
!if [ ! -d 'mjlab-custom' ]; then git clone -q https://github.com/anunpanya9/mjlab-custom.git; fi
%cd /content/mjlab-custom

import os
import sys

# ติดตั้ง uv (pip ธรรมดาอ่าน [tool.uv.sources] ของ mjlab ไม่ได้ → ลง deps ไม่ครบ)
!curl -LsSf https://astral.sh/uv/install.sh | sh
UV = "/usr/local/bin/uv"

# สำคัญ: ลงเข้า Python 'ตัวเดียวกับที่ kernel นี้ใช้' (sys.executable)
# ไม่งั้น uv จะลงเข้า /usr แล้ว kernel มองไม่เห็น → No module named mjlab
!{UV} pip install --python {sys.executable} -e . --index-strategy unsafe-best-match

# Add current directory's src folder to sys.path to ensure mjlab is found after editable install
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# ยืนยัน import ได้ (ไม่ต้อง restart)
import mjlab

print("✓ ติดตั้ง mjlab เข้า", sys.executable, "— import ได้เลย")

/content/mjlab-custom
downloading uv 0.12.12 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.13.15 environment at: /usr
Resolved 123 packages in 531ms
Prepared 1 package in 7ms
Uninstalled 1 package in 0.64ms
Installed 1 package in 3ms
 ~ mjlab==1.6.0 (from file:///content/mjlab-custom)
✓ ติดตั้ง mjlab เข้า /usr/bin/python3 — import ได้เลย


## 3) ปิด Weights & Biases (ให้เทรนได้เลยไม่ต้อง login)

mjlab ใช้ W&B log การเทรน. ตั้ง offline เพื่อข้ามการ login (ผลเทรนยังเซฟใน
เครื่องปกติ). ถ้าอยากดู dashboard ออนไลน์ ให้ `!wandb login` แทน

In [8]:
!wandb offline

wandb: Updated settings file /content/mjlab-custom/wandb/settings
W&B offline. Running your script from this directory will only write metadata locally. Use `wandb disabled` to completely turn off W&B.


## 4) เทรน! (นี่คือหัวใจ)

**คำสั่งเดียวจบ:** เรียก train script พร้อมพารามิเตอร์:
- `Mjlab-Balance-Unitree-G1` — task ที่จะเทรน (G1 เดินตามคำสั่ง)
- `--env.scene.num-envs 2048` — จำลอง 2048 ตัวขนาน (ยิ่งเยอะยิ่งเรียนเร็ว
  แต่กิน VRAM; T4 ไหว ~2048–4096)
- `--agent.max-iterations 300` — เทรน 300 รอบ **(ตัวอย่างให้เห็นผลไว ~10-20
  นาที)**. ผลจริงจังใช้ 3000+ รอบ (เดินสวยขึ้นมาก แต่นานขึ้น)
- `--agent.save-interval 50` — เซฟ checkpoint ทุก 50 รอบ

**ระหว่างเทรน** ดูค่า `Mean reward` ใน log — ควร**ค่อยๆ เพิ่มขึ้น** นั่นคือ
สัญญาณว่า policy กำลังเรียนรู้ที่จะเดินตามคำสั่ง

## เชื่อม Google Drive (เก็บ logs/checkpoint)

เซฟ checkpoint ลง Drive โดยตรง — session หลุดก็ไม่หาย (จะมี popup ให้ allow)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
LOG_ROOT = '/content/drive/MyDrive/mjlab_logs'
os.makedirs(LOG_ROOT, exist_ok=True)
print('checkpoint จะเซฟที่:', LOG_ROOT)

In [9]:
import os
os.environ['LOG_ROOT_ENV'] = LOG_ROOT

!python -m mjlab.scripts.train Mjlab-Balance-Unitree-G1 \
    --env.scene.num-envs 2048 \
    --agent.max-iterations 3000 \
    --agent.save-interval 50 \
    --agent.logger tensorboard \
    --log-root $LOG_ROOT_ENV

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
             Metrics/slip_velocity_mean: 0.0585
             Metrics/landing_force_mean: 167.1391
   Episode_Reward/track_linear_velocity: 0.0000
  Episode_Reward/track_angular_velocity: 0.0000
                 Episode_Reward/upright: 1.9859
                    Episode_Reward/pose: 1.7078
            Episode_Reward/body_ang_vel: -0.0172
        Episode_Reward/angular_momentum: -0.0212
          Episode_Reward/dof_pos_limits: 0.0000
          Episode_Reward/action_rate_l2: -0.2601
                Episode_Reward/air_time: 0.0000
          Episode_Reward/foot_clearance: 0.0000
       Episode_Reward/foot_swing_height: 0.0000
               Episode_Reward/foot_slip: 0.0000
            Episode_Reward/soft_landing: 0.0000
         Episode_Reward/self_collisions: 0.0000
        Episode_Metrics/mean_action_acc: 0.3874
   Curriculum/command_vel/lin_vel_x_min: -1.0000
   Curriculum/command_vel/lin_vel_x_max: 1.0000
   Curriculum/command_vel

## 5) หา checkpoint ที่เทรนได้ (ไฟล์โมเดล .pt)

mjlab เซฟ checkpoint ที่ `logs/rsl_rl/<experiment_name>/<run>/`. สำหรับ G1
velocity ชื่อ experiment คือ `g1_balance`. เราหา run ล่าสุดและ checkpoint
รอบสูงสุด

In [10]:
import os
from pathlib import Path

log_dir = Path(LOG_ROOT) / 'g1_balance'
runs = sorted(log_dir.glob('*'), key=os.path.getmtime, reverse=True)
assert runs, 'ไม่พบ run บน Drive'
latest = runs[0]
ckpts = sorted(latest.glob('model_*.pt'),
               key=lambda p: int(''.join(filter(str.isdigit, p.stem))))
checkpoint = str(ckpts[-1])
print('✅ โมเดล:', checkpoint, '(บน Google Drive)')

run ล่าสุด : 2026-09-10_02-58-17
checkpoints: ['model_0.pt', 'model_50.pt', 'model_100.pt', 'model_150.pt', 'model_200.pt', 'model_250.pt', 'model_300.pt', 'model_350.pt', 'model_400.pt', 'model_450.pt', 'model_500.pt', 'model_550.pt', 'model_600.pt', 'model_650.pt', 'model_700.pt', 'model_750.pt', 'model_800.pt', 'model_850.pt', 'model_900.pt', 'model_950.pt', 'model_1000.pt', 'model_1050.pt', 'model_1100.pt', 'model_1150.pt', 'model_1200.pt', 'model_1250.pt', 'model_1300.pt', 'model_1350.pt', 'model_1400.pt', 'model_1450.pt', 'model_1500.pt', 'model_1550.pt', 'model_1600.pt', 'model_1650.pt', 'model_1700.pt', 'model_1750.pt', 'model_1800.pt', 'model_1850.pt', 'model_1900.pt', 'model_1950.pt', 'model_2000.pt', 'model_2050.pt', 'model_2100.pt', 'model_2150.pt', 'model_2200.pt', 'model_2250.pt', 'model_2300.pt', 'model_2350.pt', 'model_2400.pt', 'model_2450.pt', 'model_2500.pt', 'model_2550.pt', 'model_2600.pt', 'model_2650.pt', 'model_2700.pt', 'model_2750.pt', 'model_2800.pt', 'model_

## 6) ทดสอบโมเดล — ให้ policy ที่เทรนแล้วสั่งหุ่น แล้วอัดวิดีโอ

**แนวคิด:** โหลด checkpoint กลับเข้ามาเป็น policy แล้วให้มันสั่งหุ่นจริง (ไม่ใช่
random แล้ว!) เราเรนเดอร์ทีละเฟรมเองเป็นวิดีโอ — วิธีนี้ควบคุมได้เต็มที่และ
**ไม่ค้าง** (ไม่เปิด viewer ที่ Colab ไม่มีจอ)

> ตั้ง `MUJOCO_GL=egl` เพื่อเรนเดอร์แบบ headless (ไม่ต้องมีจอ) — ต้องตั้ง
> **ก่อน** import mujoco/สร้าง env ครั้งแรก

In [11]:
os.environ["MUJOCO_GL"] = "egl"  # headless render บน Colab

from dataclasses import asdict

import imageio
import torch

import mjlab.tasks  # noqa: F401
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.rl.runner import MjlabOnPolicyRunner
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls

TASK = "Mjlab-Balance-Unitree-G1"
device = "cuda" if torch.cuda.is_available() else "cpu"

# สร้าง env แบบ play (1 ตัว) พร้อม render_mode
env_cfg = load_env_cfg(TASK, play=True)
env_cfg.scene.num_envs = 1
eval_env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode="rgb_array")

# โหลด policy จาก checkpoint
agent_cfg = load_rl_cfg(TASK)
runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
wrapped = RslRlVecEnvWrapper(eval_env, clip_actions=agent_cfg.clip_actions)
runner = runner_cls(wrapped, asdict(agent_cfg), device=device)
runner.load(checkpoint, load_cfg={"actor": True}, strict=True, map_location=device)
policy = runner.get_inference_policy(device=device)
print("✓ โหลด policy จาก", Path(checkpoint).name)

Warp 1.17.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.17.0
Module mujoco_warp._src.smooth b9cc505 load on device 'cuda:0' took 4.04 ms  (cached)
Module _nxn_broadphase__locals__kernel_3a0c7ab9 3a0c7ab load on device 'cuda:0' took 0.63 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_8d707f29 8d707f2 load on device 'cuda:0' took 0.84 ms  (cached)
Module mujoco_warp._src.constraint 9435a53 load on device 'cuda:0' took 0.48 ms  (cached)
Module _friction_dof__locals__kernel_85239eb5 85239eb load on device 'cuda:0' took 0.61 ms  (cached)
Module _limit_slide_hinge__locals__kernel_b8014c9f b8014c9 load on device 'cuda:0' took 0.58 ms  (cached)
Module _efc_contact_init__locals__kernel_34eca94d 34eca94 load on device 'cuda:0' took 0.94 ms  (cached)
Module _efc_contact_jac_sparse__locals__kernel_eb752226 eb75222 load on devic

In [12]:
# rollout: ให้ policy สั่งหุ่น 200 step แล้วเก็บเฟรมเป็นวิดีโอ
obs = wrapped.get_observations()
frames = []
for step in range(200):
  with torch.inference_mode():
    action = policy(obs)
  obs, _, _, _ = wrapped.step(action)
  frames.append(eval_env.render())  # rgb array (H, W, 3)

VIDEO_PATH = os.path.join(LOG_ROOT, 'g1_balance.mp4')
out = VIDEO_PATH
imageio.mimsave(out, frames, fps=30)
print(f"✓ อัดวิดีโอ {len(frames)} เฟรม -> {out}")

✓ อัดวิดีโอ 200 เฟรม -> /content/g1_balance.mp4


In [13]:
from IPython.display import Video

Video(VIDEO_PATH, embed=True, width=480)

## 7) ⬇️ ดาวน์โหลดโมเดลไปใช้งาน

ไฟล์ `.pt` นี้คือ **โมเดลที่ใช้งานได้จริง** — เอาไปโหลดที่เครื่องอื่น
(ที่มี mjlab) แล้วสั่งหุ่นด้วย `play --checkpoint-file <ไฟล์>` ได้เลย

In [14]:
from google.colab import files

print("กำลังดาวน์โหลด:", checkpoint)
files.download(checkpoint)

กำลังดาวน์โหลด: /content/mjlab-custom/logs/rsl_rl/g1_balance/2026-09-10_02-58-17/model_2999.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8) สรุป + ทำต่อ

คุณเพิ่ง**เทรนโมเดล RL จริง**และได้ไฟล์ `.pt` ไปใช้งาน 🎉

**เอาโมเดลไปใช้ที่เครื่องตัวเอง** (ที่มี mjlab):
```bash
uv run play Mjlab-Balance-Unitree-G1 --checkpoint-file model.pt
```

**อยากให้หุ่นเดินสวยขึ้น?** เพิ่ม `--agent.max-iterations` เป็น 3000–10000
(นานขึ้นแต่ผลดีขึ้นมาก) แล้วเทรนใหม่

**อยากเทรนงานหยิบของแทน?** เปลี่ยน task เป็น `Mjlab-Lift-Cube-G1`
(experiment_name = `g1_lift_cube`) แล้วแก้ path ใน cell ที่ 5 ตามนั้น

**เข้าใจว่าข้างในทำงานยังไง?** กลับไปดู `explore_g1_balance.ipynb` และ
`explore_g1_manipulation.ipynb` ที่แกะ MDP ทีละส่วน